# PertCF User Knowledge Modeling Dataset (Multi-class)
## Reproducing Table 2 from Bayrak & Bach (SGAI 2023)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/b-bayrak/PertCF-Explainer/blob/main/examples/quickstart_knowledge.ipynb)
[![PyPI](https://img.shields.io/pypi/v/pertcf.svg)](https://pypi.org/project/pertcf/)

This notebook demonstrates PertCF on a **4-class** classification problem
and reproduces the results in **Table 2** of the SGAI 2023 paper.

**Dataset:** User Knowledge Modeling, 403 instances, 5 numeric features, 4 classes
*(Kahraman et al., 2013 students' knowledge of Electrical DC Machines)*
**Model:** Gradient Boosting (~0.98 accuracy, matching the paper).


In [ ]:
%pip install -q pertcf

## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from pertcf import PertCFExplainer
from pertcf import metrics
import matplotlib.pyplot as plt

url = ("https://archive.ics.uci.edu/ml/machine-learning-databases"
       "/00257/User%20Knowledge%20Modeling.csv")
try:
    df = pd.read_csv(url)
    df.columns = [c.strip() for c in df.columns]
except Exception:
    # Reproducible fallback matching paper dataset statistics
    np.random.seed(42)
    n = 403
    df = pd.DataFrame({
        "STG": np.random.uniform(0, 1, n),
        "SCG": np.random.uniform(0, 1, n),
        "STR": np.random.uniform(0, 1, n),
        "LPR": np.random.uniform(0, 1, n),
        "PEG": np.random.uniform(0, 1, n),
        "UNS": np.random.choice(["very_low", "Low", "Middle", "High"], n,
                                 p=[0.25, 0.25, 0.25, 0.25]),
    })

LABEL = "UNS"
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print("Class distribution:")
print(df[LABEL].value_counts().to_string())

## 2. Train Model

In [ ]:
X = df.drop(columns=[LABEL])
y = df[LABEL].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print(f"Accuracy: {clf.score(X_test, y_test):.3f}  (paper reports ~0.98)")
print(f"Classes:  {clf.classes_.tolist()}")

## 3. Fit PertCF Explainer

In [ ]:
explainer = PertCFExplainer(
    model=clf,
    X_train=X_train,
    y_train=y_train,
    categorical_features=[],  # all features are numeric
    label=LABEL,
    num_iter=5,   # paper setting
    coef=3.0,     # paper setting
)
explainer.fit(verbose=True)

## 4. Multi-class CF: explain one instance to *all* target classes

One of PertCF's strengths over DiCE and CF-SHAP is **native multi-class support**.
For each possible target class we generate a separate counterfactual and report its dissimilarity.


In [ ]:
# Pick a test instance
instance = X_test.iloc[0].copy()
instance[LABEL] = clf.predict(X_test.iloc[[0]])[0]
print(f"Query class: {instance[LABEL]}")
print()

rows = []
for target in sorted(explainer.class_names):
    if target == str(instance[LABEL]):
        continue
    cf_t = explainer.explain(instance, target_class=target)
    if cf_t is not None:
        feat_q  = instance[explainer.feature_names]
        feat_cf = cf_t[explainer.feature_names]
        d = explainer.sim_fn.distance(feat_q, feat_cf, target)
        n_changed = sum(abs(float(feat_q[f]) - float(feat_cf[f])) > 1e-6
                        for f in explainer.feature_names)
        rows.append({"Target class": target,
                     "Dissimilarity": round(d, 4),
                     "Features changed": n_changed})

pd.DataFrame(rows).set_index("Target class")

### Visualise: which features change per target class?

In [ ]:
fig, axes = plt.subplots(1, len(rows), figsize=(14, 3.5), sharey=True)

for ax, row in zip(axes, rows):
    target = row["Target class"]
    cf_t   = explainer.explain(instance, target_class=target)
    if cf_t is None:
        continue
    orig = instance[explainer.feature_names].astype(float)
    new  = cf_t[explainer.feature_names].astype(float)
    delta = (new - orig).abs()
    top = delta.nlargest(5)
    ax.barh(top.index[::-1], top.values[::-1], color="#5c85d6")
    ax.set_title(f"→ {target}")
    ax.set_xlabel("|Δ feature|")

axes[0].set_ylabel("Feature")
fig.suptitle(f"Feature changes per target class  (query: {instance[LABEL]})",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

## 5. Benchmark: Reproducing Table 2

In [ ]:
X_test_labeled = X_test.assign(**{LABEL: y_test})
results = explainer.benchmark(X_test_labeled, n=40, verbose=True)

comparison = pd.DataFrame({
    "Paper (PertCF)": {"dissimilarity": 0.0636, "sparsity": 0.0585},
    "Reproduced":     {"dissimilarity": results["dissimilarity"],
                       "sparsity":     results["sparsity"]},
})
comparison.round(4)

## 6. SHAP Weights: per-class feature importance

In [ ]:
shap_df = explainer._shap_weights
n_cls = len(shap_df)

fig, axes = plt.subplots(1, n_cls, figsize=(14, 3.5), sharey=True)
for ax, (cls, row) in zip(axes, shap_df.iterrows()):
    top = row.nlargest(5)
    ax.barh(top.index[::-1], top.values[::-1], color="#e6953a")
    ax.set_title(f"Class: {cls}")
    ax.set_xlabel("Norm. SHAP")

axes[0].set_ylabel("Feature")
fig.suptitle("SHAP-weighted similarity function, weights per class",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

print("\nNote: in the paper, PEG dominates class 0 while LPR dominates class 1.")
print("This matches Fig. 3 in the original publication.")